<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/demos/drone_gridworld_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drone path planning
*(This problem is adapted from Stanford AA203 course)*

In this problem, we consider a rescue drone that needs to deliver aid to a goal state while avoiding regions with fire and uncertain wind conditions.

The world is represented as an $n \times n$ grid, i.e., the state space is

$$ \mathcal{S}  := \lbrace(x_1, x_2) \in \mathbb{Z}_+^2 \mid | x_1, x_2 \in \lbrace 0, 1 . . . , n − 1\rbrace\rbrace \cup \lbrace (\texttt{None}, \texttt{None}) \rbrace, .$$

In these coordinates, $(0, 0)$ represents the bottom left corner of the map and $(n−1, n−1)$ represents the top right corner of the map. While $(\texttt{None}, \texttt{None})$ is a terminal state. For any non-terminal state, from any location $x = (x_1, x_2) \in \mathcal{S}$, the drone has four possible directions it can move in, i.e.,

$$ \mathcal{A}:= \lbrace \texttt{up}, \texttt{down}, \texttt{left}, \texttt{right} \rbrace.$$

The corresponding state changes for each action are:
- $\texttt{up}: (x_1, x_2) \mapsto (x_1, x_2+1)$
- $\texttt{down}: (x_1, x_2) \mapsto (x_1, x_2-1)$
- $\texttt{left}: (x_1, x_2) \mapsto (x_1-1, x_2)$
- $\texttt{right}: (x_1, x_2) \mapsto (x_1+1, x_2)$

There is a storm centered at $x_\mathrm{eye} \in \mathcal{S}$. The storm’s influence is strongest at its center and decays farther from the center according to the equation

$$ \omega(x) = \exp\biggl( -\frac{\| x - x_\mathrm{eye}\|_2^2}{2\sigma^2}\biggr)$$

Given its current state $x$ and action $a$, the drone’s next state is determined as follows:
- With probability $\omega(x)$, the storm will cause the drone to move in a uniformly random direction.
- With probability $1 − \omega(x)$, the drone will move in the direction specified by the action.
- If the resulting movement would cause the drone to leave $\mathcal{S}$, then it will not move at all. For example, if the drone is on the right boundary of the map, then moving right will do nothing.
- If the drone reaches the goal state, then the drone will always transition to a *terminal state* $(\texttt{None}, \texttt{None}).$ Once in the terminal state, the drone remains in that state indefinitely regardless of the action taken.

The drone's objective is to reach $x_\mathrm{goal} \in \mathcal{S}$. If the drone reaches the goal state, then it receives a reward of $r_\mathrm{goal}$ (successfully delivers aid), and a reward of $r_\mathrm{travel}$ otherwise (cost of traveling one unit). Additionally, there are some states where there is a fire. If the drone reaches a state where there is a fire, then it receives a reward of $r_\mathrm{fire}$ (drone suffers damage). Once the drone is in the terminal state, it receives zero reward (i.e., mission has terminated). The reward of a trajectory in this infinite horizon problem is a discounted sum of the rewards earned in each timestep, with discount factor $\gamma \in (0, 1)$.

To find the optimal policy to reach the goal state from any starting location, we perform value iteration. Recall that the value iteration repeats the Bellman update until convergence.

$$ V(x) \leftarrow \max_{a\in\mathcal{A}} \biggl( \sum_{x^\prime \in \mathcal{S}} p(x, a, x^\prime) (R(x^\prime) + \gamma V(x^\prime))\biggr) $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
import functools


In [ ]:
def is_terminal_state(state):
    """
    Check if the state is a terminal state.
    Args:
        state: Current state (row, column).
    Returns:
        True if the state is terminal, False otherwise.
    """
    return state == (None, None)


def _state_space(max_rows, max_columns):
    return [(i, j) for i in range(max_rows) for j in range(max_columns)] + [(None, None)]


def _reward(state, next_state, fire_states, goal_states, fire_value, goal_value, travel_value):
    """
    Reward function for the grid world.
    Args:
        state: Current state (row, column).
        fire_states: List or set of fire states.
        goal_states: List or set of goal states.
        fire_value: Reward value for fire states.
        goal_value: Reward value for goal states.
    Returns:
        Reward value for the current state.
    """
    #### FILL CODE HERE ####
    reward = 0
    if state in goal_states:
        reward += goal_value
    elif state in fire_states:
        reward += fire_value
    elif is_terminal_state(state):
        reward += 0


    if not is_terminal_state(next_state):
        reward += travel_value

    return reward
    ########################

def _transition_function(s, a, w=0,
                        max_rows=20, # number of rows
                        max_columns=20, # number of columns
                        goal_states=set([]),
                        action_set=["down", "right", "up", "left"]):
    """
    Transition function for the grid world.
    Args:
        s: Current state (row, column).
        a: Action to take.
        w: Probability of taking the action.
        max_rows: Number of rows in the grid.
        max_columns: Number of columns in the
            grid.
        action_set: List of possible actions.
    Returns:
        New state after taking the action.
    """
    i,j = s
    if is_terminal_state(s) or (s in goal_states):
        return (None, None)
    if (np.random.rand(1) < w)[0]:
        a = np.random.choice(action_set)
    if a == "up":
        return (min(i+1, max_rows-1), j)
    if a == "right":
        return (i, min(j+1, max_columns-1))
    if a == "down":
        return (max(i-1, 0), j)
    if a == "left":
        return (i, max(j-1, 0))


def _compute_omega_probability(state, storm_eye, storm_sigma, storm_active=True):
    """
    Computes the probability of a state being affected by a storm.
    Args:
        state: Current state (row, column).
        storm_eye: Center of the storm (row, column).
        storm_sigma: Standard deviation of the storm.
    Returns:
        Probability of the state being affected by the storm.
    """
    if not storm_active:
        return 0
    if is_terminal_state(state):
        return 0
    w = 0
    for (se, ss) in zip(storm_eye, storm_sigma):
        w += np.exp(-((state[0] - se[0])**2 + (state[1] - se[1])**2) / (2 * ss**2))
    return min(w, 1)

def plot_policy(policy):
    for (row, col), action in policy.items():
        if row is None or col is None:
            continue
        if action == "up":
            plt.text(col + 0.5, row + 0.5, '↑', ha='center', va='center', color='black', fontsize=8)
        elif action == "down":
            plt.text(col + 0.5, row + 0.5, '↓', ha='center', va='center', color='black', fontsize=8)
        elif action == "left":
            plt.text(col + 0.5, row + 0.5, '←', ha='center', va='center', color='black', fontsize=8)
        elif action == "right":
            plt.text(col + 0.5, row + 0.5, '→', ha='center', va='center', color='black', fontsize=8)
        elif action == "None":
            plt.text(col + 0.5, row + 0.5, '•', ha='center', va='center', color='black', fontsize=8)

def get_box_states(bottom_left, height, width):
    """Returns a list of states (row, col) within a rectangular box defined by the bottom left corner, height, and width."""
    states = []
    for i in range(bottom_left[0], bottom_left[0] + height):
        for j in range(bottom_left[1], bottom_left[1] + width):
            states.append((i, j))
    return states

def plot_value_function(value):
    max_columns, max_rows = value.shape[1], value.shape[0]
    plt.imshow(value, origin='lower', extent=[0, max_columns, 0, max_rows], cmap='jet', interpolation='nearest')
    plt.colorbar(label='Value')
    plt.title('Value Function')
    plt.xlabel('Column')
    plt.ylabel('Row')
    plt.xticks(ticks=np.arange(0.5, max_columns, 1), labels=np.arange(0, max_columns))
    plt.yticks(ticks=np.arange(0.5, max_rows, 1), labels=np.arange(0, max_rows))

def plot_value_function_numbers(value):
    max_columns, max_rows = value.shape[1], value.shape[0]
    for i in range(max_rows):
        for j in range(max_columns):
            plt.text(j + 0.5, i + 0.5, f'{value[i, j]:.1f}', ha='center', va='center', color='black', fontsize=8)

In [ ]:
# problem set up
# size of grid world
max_rows, max_columns = 20, 20

# choose where the fires are
fire_states = set(get_box_states((9,9), 4, 4) + get_box_states((13, 3), 3, 3))
# fire_states = set([])

# define storm parameters: on/off, location, and spread
storm_active = True
storm_eye = [(10, 6)]
storm_sigma = [10]

# define goal states
goal_states = set([(19,9)])

# define discount factor and rewards
gamma = 1.0
fire_value = -2
goal_value = 2
travel_value = -0.1

# action set
action_set=["down", "right", "up", "left"]


# fix the problem parameters in the functions to avoid passing them every time
state_space = functools.partial(_state_space, max_rows=max_rows, max_columns=max_columns)
reward = functools.partial(_reward, fire_states=fire_states, goal_states=goal_states, fire_value=fire_value, goal_value=goal_value, travel_value=travel_value)
transition_function = functools.partial(_transition_function, max_rows=max_rows, max_columns=max_columns, goal_states=goal_states, action_set=action_set)
compute_omega_probability = functools.partial(_compute_omega_probability, storm_eye=storm_eye, storm_sigma=storm_sigma, storm_active=storm_active)



In [ ]:
# defines functions: transition probability, get_next_state, and simulate
def probability_function(state, action, next_state, w,
                          action_set=["down", "right", "up", "left"]):
    """
    Computes the probability of transitioning to a next state given the current state and action.
    Args:
        state: Current state (row, column).
        action: Action to take.
        next_state: Next state (row, column).
        w: Probability of taking random action.
        action_set: List of possible actions.
    Returns:
        Probability of transitioning to the next state.
    """

    #### FILL CODE HERE ####
    # HINT: Our solution takes ~3 lines of code
    deterministic_next_state = transition_function(state, action, w=0)
    random_probability = sum(transition_function(state, a, w=0) == next_state for a in action_set) * w / len(action_set)

    return random_probability + (1 - w) * (deterministic_next_state == next_state)
    # return (transition_function(state, action, w=0) == next_state) * 1. # UPDATE THIS LINE

def get_possible_next_states(state, action_set):
    """
    Returns the set of possible next states given the current state.
    Args:
        state: Current state (row, column).
    Returns:
        Set of possible next states.
    """
    return set([transition_function(state, action, w=0) for action in action_set])

def simulate(start_state, policy, num_steps):
    """
    Simulates the agent's trajectory in the grid world.
    Args:
        start_state: Starting state (row, column).
        policy: Policy to follow.
        num_steps: Number of steps to simulate.
    Returns:
        List of states visited during the simulation.
    """
    states = [start_state]
    for _ in range(num_steps):
        action = policy[start_state]
        w = compute_omega_probability(start_state)
        next_state = transition_function(start_state, action, w=w)
        if is_terminal_state(next_state):
            break
        start_state = next_state
        states.append(start_state)
    return states

In [ ]:
def bellman_update(value_tuple, gamma, action_set):
    """
    Performs a Bellman update on the value function.
    Args:
        value_tuple: Current value function. A tuple of (value, value_terminal).
        value: Array representing the value at each state in the grid
        value_terminal: Value of the terminal state.
        gamma: Discount factor.
        action_set: List of possible actions.
    Returns:
        Updated value_tuple and policy as a dictionary.
    """
    value, value_terminal = value_tuple
    new_value = value.copy()
    policy = {}
    for state in state_space():
        w = compute_omega_probability(state)
        qs = []
        for action in action_set:
            expected_value = 0
            for next_state in get_possible_next_states(state, action_set):
                if is_terminal_state(next_state):
                    val = value_terminal
                else:
                    val = value[next_state]
                expected_value += probability_function(state, action, next_state, w=w) * (reward(state, next_state) + gamma * val)
                if not isinstance(expected_value, float):
                    print("Debug: expected_value is float at state {}, action {}, next_state {}".format(state, action, next_state))
            qs.append(expected_value)
        if is_terminal_state(state):
            new_value_terminal = max(qs)
            policy[state] = "None"
        else:
            new_value[state] = max(qs)
            policy[state] = action_set[np.argmax(qs)]
    return (new_value, new_value_terminal), policy


In [ ]:
# Initialize the value function
num_iterations = 150 # feel free to change this value as needed

V0 = np.zeros([max_rows, max_columns])
for fs in fire_states:
    V0[fs] = fire_value
for gs in goal_states:
    V0[gs] = goal_value
V = (V0, 0)
# keep list of value functions
Vs = [V]
dV = []
policies = []
for _ in range(num_iterations):
    # perform Bellman update
    V_new, policy = bellman_update(V, gamma, action_set)
    # store the new value function
    Vs.append(V_new)
    policies.append(policy)
    dV.append(np.abs(V_new[0] - V[0]).max())

    # check for convergence
    if np.abs(V_new[0] - V[0]).max() < 1e-3:
        print("Converged!")
        break
    # update the value function
    V = V_new


In [ ]:
plt.figure(figsize=(4,2))
plt.plot(dV)
plt.title('Convergence of Value Function')
plt.xlabel('Iteration')
plt.ylabel('Max Change in Value Function')
plt.grid()

In [ ]:
# visualize the value function and storm strength

simulate_trajectory = True

if simulate_trajectory:
    start_state = (0,8) # pick a starting state
    num_steps = 200 # feel free to change this value as needed
    # simulate the trajectory
    trajectory = simulate(start_state, policy, num_steps)

if storm_active:
    # compute the storm strength for each state for plotting later
    storm_strength = np.zeros([max_rows, max_columns])
    for state in state_space():
        if not is_terminal_state(state):
            storm_strength[state] = compute_omega_probability(state)

@interact(iteration=(0,len(Vs)-1), t=(0,len(trajectory)-1), second_plot=['Value Function', 'Storm Strength'])
def plot_results(iteration, t, second_plot):
    plt.figure(figsize=(20, 8))
    plt.subplot(1, 2, 1)
    plot_value_function(Vs[iteration][0])
    if storm_active:
        for se in storm_eye:
            plt.scatter(se[1] + 0.5, se[0] + 0.5, c='magenta', s=100)
        plt.scatter(se[1] + 0.5, se[0] + 0.5, c='magenta', s=100, label='Storm Eye')
    if len(fire_states) > 0:
        for fire_state in fire_states:
            plt.scatter(fire_state[1] + 0.5, fire_state[0] + 0.5, c='red', s=100)
        plt.scatter(fire_state[1] + 0.5, fire_state[0] + 0.5, c='red', s=100, label='Fire State')
    for goal_state in goal_states:
        plt.scatter(goal_state[1] + 0.5, goal_state[0] + 0.5, c='green', s=100)
    plt.scatter(goal_state[1] + 0.5, goal_state[0] + 0.5, c='green', s=100, label='Goal State')

    # Overlay the policy
    if iteration > 0:
        plot_policy(policies[iteration-1])
    if simulate_trajectory:
        # Plot the trajectory
        trajectory_x = [state[1] + 0.5 for state in trajectory]
        trajectory_y = [state[0] + 0.5 for state in trajectory]
        plt.plot(trajectory_x, trajectory_y, color='C0', label='Trajectory', linewidth=2)
        plt.scatter(trajectory_x[t], trajectory_y[t], color='C0', s=100, label='Current State')
    plt.legend(loc="lower left", framealpha=0.6, ncols=3)

    plt.subplot(1, 2, 2)
    if second_plot == 'Storm Strength':
        plt.imshow(storm_strength, origin='lower', extent=[0, max_columns, 0, max_rows], cmap='jet', interpolation='nearest')
        plt.colorbar(label='Storm Strength')
        plt.title('Storm Strength')
        plt.xlabel('Column')
        plt.ylabel('Row')
        plt.xticks(ticks=np.arange(0.5, max_columns, 1), labels=np.arange(0, max_columns))
        plt.yticks(ticks=np.arange(0.5, max_rows, 1), labels=np.arange(0, max_rows))
    elif second_plot == 'Value Function':
        plot_value_function(Vs[iteration][0])
        plot_value_function_numbers(Vs[iteration][0])

    plt.show()